In [23]:
import os
import sys
import anndata as ad
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import re
import gseapy as gp
import anndata as ad
import statistics
import tempfile
import sklearn
import cosg
import leidenalg
import celltypist
import muon as mu
from tqdm import tqdm
sc._settings.ScanpyConfig.n_jobs= 24
sc.settings.verbosity = 1
# Adjust Scanpy figure defaults
sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=400,
    facecolor = 'white', figsize=(8,8), format='png')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

In [24]:
def check_dict_duplicates(dict):
    seen = set()
    dup = any(item in seen or seen.add(item) for lst in dict.values() for item in lst)
    if dup==False:
        return '无重复'
    else:
        return '有重复'

In [25]:
os.environ["R_HOME"] = "/home/liyanguo/anaconda3/envs/R/lib/R/"
import rpy2.rinterface_lib.callbacks
import anndata2ri
import logging

from rpy2.robjects import pandas2ri
from rpy2.robjects import r

sc.settings.verbosity = 0
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [26]:
%%R
suppressPackageStartupMessages({
    library(Seurat)
    library(clusterProfiler)
    library(ggplot2)
    library(dplyr)
    library(anndata)
    library(SingleCellExperiment)
    library(reticulate)
})

In [27]:
obj_path = '/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R7/'

In [28]:
finnal_path = '/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Finnal/'

In [29]:
leiden_groups=['L4_leiden_TOTALVI_0.1','L4_leiden_TOTALVI_0.5', 'L4_leiden_TOTALVI_1', 'L4_leiden_TOTALVI_1.5', 'L4_leiden_TOTALVI_0.3','L4_leiden_TOTALVI_0.8']

In [30]:
def get_norm_annotation_data(celltype):
    adata = sc.read_h5ad(f"{obj_path}/{celltype}/{celltype}_count_scRNA.h5ad")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adt = sc.read_h5ad(f"{obj_path}/{celltype}/{celltype}_preprocess_scADT.h5ad")
    adata.obsm = adt.obsm
    
    # read Level1, tcr, bcr infor
    indices = pd.read_csv(f'{obj_path}/{celltype}/R7_indices_{celltype}.csv',index_col=0)
    # join leiden groups
    leiden_data = adt.obs.loc[:,leiden_groups]
    adata.obs = adata.obs.join(indices, how='left')
    adt.obs = adt.obs.join(indices, how='left')
    adata.obs = adata.obs.join(leiden_data, how='left')
    return adata,adt,leiden_data

# 1. Th1 Cell Refine

In [14]:
celltype="Th1"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [630]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4,
           )

In [631]:
groupby = "L4_leiden_TOTALVI_1.5"

In [251]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [253]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='obs',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
sc.pl.umap(adt, color='HLA-DR',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',layer='dsb')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='CD183',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD185',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False,layer='dsb')
sc.pl.umap(adt, color='CD196',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False,layer='dsb')
sc.pl.umap(adt, color='CD161',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False,layer='dsb')
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD16',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False,layer='dsb')
sc.pl.umap(adt, color='IgM',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False,layer='dsb')
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='Celltype_L4_L5_Refine_R2',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='CCR10',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adata, color='MKI67',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
sc.pl.umap(adata, color='F5',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB',
                      'PDCD1','CTLA4','HAVCR2','LAG3','AHR','STAT1','GATA3','FOXP3','CD38','CCR5','TIGIT','ENTPD1','CD160',
                     'TCF7','TBX21','EOMES'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB',
                      'LTK','PTPN13','PDE4D','CCR6','RORC','NR1D1','CTSH','KIF5C','LGALS3','USP10','CMTM6','TOB1',
                     'TNFSF13B','CISH','AQP3','AUTS2','NSG1','S100A4'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1/Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'GZMH','IL18RAP','S1PR5','LYAR','NKG7','CST7','PRF1','TBX21','LINC01871','KLRG1','MYBL1','EOMES',
                     'EFHD2','DUSP2','SAMD3','CTSW','ID2','MATK','HOPX',],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','TBX21','IFNG',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th22
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'AHR',
                      'CRIP1','LGALS1','LGALS3','S100A10','S100A4','PI16','LMNA','ANXA5','ANXA2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th2
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'PTGDR2','SNED1','NEFL','GATA3','FXYD7','C1orf162','GDPD5','IL4R','CAPG',
                     'LGALS1','TNFSF10','TNFRSF4','PPP1R9B','CSGALNACT1','NIBAN1','ERN1','SORL1','RUNX2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th from BD
#TNFSF8=CD30L B3GAT1=CD57  BTLA= CD272  SLAMF5=CD84 HAVCR2=CD365
sc.pl.dotplot(adata, ['CXCR5','IL6R','TNFSF8','NRP1','IL21R','B3GAT1','BCL6','MAF','STAT3','ICOS','PDCD1','TIGIT','BTLA','CD200','SLAMF1','CD84',#Tfh
                      'IL4','IL17F','IL17A','IL21',#tfh分泌
                      'GATA3','SMAD1','STAT6','SPI1','IRF4',#Th9
                      'IL9','IL10','CCL17','CCL22','TGFB1',#th9分泌
                      'HAVCR2','CXCR4','CCR3','CCR4','CCR8','PTGDR2','GATA3','STAT5A','STAT6','MAF','GFI1','IRF4','NOTCH1','NOTCH2','IL1RL1','IL17RB','IFNGR1','IFNGR2','TNFRSF8',#Th2
                      'IL2','IL5','IL6','IL10','IL13','IL31',#Th2分泌
                      'CCR4','CCR6','CCR10','AHR','PDGFRA','PDGFRB',#Th22
                      'IL22','TNF',#Th22分泌
                      'CXCR3','CCR5','KLRD1','TBX21','STAT1','STAT4','EOMES','RUNX3','FASLG','IL12RB1','IL12RB2','IL18R1','IL27RA','NOTCH3','TNFSF11','ICOS','HAVCR2','DPP4',#Th1
                      'LTB','LTA','PRF1','GZMB','GZMA','TNF','IFNG',#Th1分泌
                      'CCR4','CCR6','KLRB1','ICOS','HAVCR2','RORC','RORA','STAT3','RUNX1','BATF','IRF4','MAF','IL6R','IL13RA1','IL21R','IL23R',#Th17
                      'TNF','CCL20','IL17A','IL17F','IL21','IL22','IL24','IL26',#Th17分泌
                      ],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs[groupby]=="6",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [632]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_1.5
13    29776
14    23315
9     20772
8     19639
7     18239
6     18195
12    17372
11    16055
4     15614
0     13162
3      8409
2      3199
10     1114
5      1022
15      668
1       133
16      102
17        9
18        8
19        5
Name: count, dtype: int64

In [633]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#'Th22':#CCR4+ CCR6+ CCR10+ KLRB1- GZMK- GATA3lo CCL5- GATA3+,CCR4+ CCR6+ KLRB1- GZMKlo AHRlo
#'Th1/Th17':# CCR6+ CCR4- CXCR3+ KLRB1+ GZMK+ CCL5+
#'Th17':#step1 CCR6+ CCR4- KLRB1+ GZMK- CCL5- RORC+ GZMK- TBX21-
#'Th2':#step1 CCR4+ CCR6- KLRB1- GATA3+ GZMK- CCL5-；CCR4+ CXCR5- GATA3+ CCL5-.CD45RA- CD279- TBX21- LEF1+ ,且没有CCL5- GZMH- GZMK- GZMB- KLRB1-各种标记的情况下CD62L+ CD27+ CD25+ 
#'Th1':#step3 确认 CXCR3+ CCR6- KLRB1- GZMK+ CD279+ GZMK+ CCL5+. CD279+(核心) GZMK+(核心) TBX21+ CCL5+(核心) TIGIT+ KLRB1-(核心) CD127-/IL7Rlo CCR7-CD197- SELLlo/CD62Llo  GZMH- GNLY- PRF1- GZMB- 的是Th1

cell_dict = {'Th22':['12',],#CCR4+ CCR6+ CCR10+ KLRB1lo/- GZMK- AHR+   CRIP1 LGALS1 LGALS3 PI16 ANXA5 ANXA2   https://www.sciencedirect.com/science/article/pii/S1933021922002033
             #'Th2|Th22':[''],#PTGDR2=CRTH2 GATA3++ IL4R+ CCR4+ CCR6- KLRB1- GZMK- CCL5- CD62L+ CD25+     PTGDR2,SNED1,NEFL,GATA3,FXYD7,C1orf162
             'Th1':['6','8','9','10','13','14','15','16'],#EOMES+ GZMK+ CCL5+ KLRB1- CXCR3+ CCR6- CD279+ TBX21+ IFNG+.  CMC1,CST7,FCRL3,CCL4,SLAMF7,EOMES,PDCD1,NKG7,CCR5,KLRK1,F2R,PLEK
             #'Th17':[],#CCR6=CD196++ RORC+ CCR4- KLRB1+ GZMK- CCL5- ICOS+    高表达LTK,PTPN13,PDE4D,CCR6,RORC,NR1D1,CTSH,KIF5C,LGALS3,USP10,CMTM6,TOB1
             'Th1/Th17':['0','3','4','7',],#DPP4+ CCR6+ EOMESlo CCR6lo CCR4- CXCR3+ KLRB1+ GZMKlo/+ CCL5+ TBX21+ 与Th1相似, 差异基因中等表达 https://rupress.org/jem/article/211/1/89/41376/Pro-inflammatory-human-Th17-cells-selectively
             #'HLA-DRhi memory':[,],#增殖特性
             'Doublet|Lowquality':['2',#CD8
                                  '5',#CD16
                                   '1','17','18','19',#Too few cell and lncRNA
                                  ],
             'Proliferative help memory CD4+ T':['11'],#TRIB2 MKI67 CD38
            }

# 0 3 4 7的疑问是，表达KLRB1 CXCR3 GZMK CCL5，但是不表达CCR6 CCR4

In [634]:
check_dict_duplicates(cell_dict)

'无重复'

In [635]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R4'] = i

In [636]:
(adata.obs['Celltype_L4_L5_Refine_R4'].isna()).value_counts()

Celltype_L4_L5_Refine_R4
False    206808
Name: count, dtype: int64

In [637]:
adata.obs['Celltype_L4_L5_Refine_R4'].value_counts()

Celltype_L4_L5_Refine_R4
Th1                                 113581
Th1/Th17                             55424
Th22                                 17372
Proliferative help memory CD4+ T     16055
Doublet|Lowquality                    4376
Name: count, dtype: int64

In [638]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R4'] != "Doublet|Lowquality",:]

In [639]:
adata.obs['Celltype_L4_L5_Refine_R4'].value_counts()

Celltype_L4_L5_Refine_R4
Th1                                 113581
Th1/Th17                             55424
Th22                                 17372
Proliferative help memory CD4+ T     16055
Name: count, dtype: int64

In [640]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine',
                           'Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3','Celltype_L4_L5_Refine_R4','receptor_type','receptor_type_BCR']]
indices.to_csv(f"{obj_path}/{celltype}/R7_refine_indices_{celltype}.csv")

# 2. Th2_Th22 T Cell Refine

In [735]:
celltype="Th2_Th22"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [736]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4,
           )

In [737]:
groupby = "L4_leiden_TOTALVI_0.8"

In [151]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [153]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='obs',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
sc.pl.umap(adt, color='HLA-DR',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',layer='dsb')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='CD183',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD185',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False,layer='dsb')
sc.pl.umap(adt, color='CD196',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False,layer='dsb')
sc.pl.umap(adt, color='CD161',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False,layer='dsb')
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD16',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False,layer='dsb')
sc.pl.umap(adt, color='IgM',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False,layer='dsb')
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='Celltype_L4_L5_Refine_R2',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='IFNG',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
sc.pl.umap(adata, color='CCR10',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
#Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LTK','PTPN13','PDE4D','CCR6','RORC','NR1D1','CTSH','KIF5C','LGALS3','USP10','CMTM6','TOB1',
                     'TNFSF13B','CISH','AQP3','AUTS2','NSG1','S100A4'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1/Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'GZMH','IL18RAP','S1PR5','LYAR','NKG7','CST7','PRF1','TBX21','LINC01871','KLRG1','MYBL1','EOMES',
                     'EFHD2','DUSP2','SAMD3','CTSW','ID2','MATK','HOPX',],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','TBX21','IFNG',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th22
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','AHR',
                      'CRIP1','LGALS1','LGALS1','S100A10','S100A4','PI16','LMNA','ANXA5','ANXA2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th2
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'PTGDR2','SNED1','NEFL','GATA3','FXYD7','C1orf162','GDPD5','IL4R','CAPG',
                     'LGALS1','TNFSF10','TNFRSF4','PPP1R9B','CSGALNACT1','NIBAN1','ERN1','SORL1','RUNX2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th from BD
#TNFSF8=CD30L B3GAT1=CD57  BTLA= CD272  SLAMF5=CD84 HAVCR2=CD365
sc.pl.dotplot(adata, ['CXCR5','IL6R','TNFSF8','NRP1','IL21R','B3GAT1','BCL6','MAF','STAT3','ICOS','PDCD1','TIGIT','BTLA','CD200','SLAMF1','CD84',#Tfh
                      'IL4','IL17F','IL17A','IL21',#tfh分泌
                      'GATA3','SMAD1','STAT6','SPI1','IRF4',#Th9
                      'IL9','IL10','CCL17','CCL22','TGFB1',#th9分泌
                      'HAVCR2','CXCR4','CCR3','CCR4','CCR8','PTGDR2','GATA3','STAT5A','STAT6','MAF','GFI1','IRF4','NOTCH1','NOTCH2','IL1RL1','IL17RB','IFNGR1','IFNGR2','TNFRSF8',#Th2
                      'IL2','IL5','IL6','IL10','IL13','IL31',#Th2分泌
                      'CCR4','CCR6','CCR10','AHR','PDGFRA','PDGFRB',#Th22
                      'IL22','TNF',#Th22分泌
                      'CXCR3','CCR5','KLRD1','TBX21','STAT1','STAT4','EOMES','RUNX3','FASLG','IL12RB1','IL12RB2','IL18R1','IL27RA','NOTCH3','TNFSF11','ICOS','HAVCR2','DPP4',#Th1
                      'LTB','LTA','PRF1','GZMB','GZMA','TNF','IFNG',#Th1分泌
                      'CCR4','CCR6','KLRB1','ICOS','HAVCR2','RORC','RORA','STAT3','RUNX1','BATF','IRF4','MAF','IL6R','IL13RA1','IL21R','IL23R',#Th17
                      'TNF','CCL20','IL17A','IL17F','IL21','IL22','IL24','IL26',#Th17分泌
                      ],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs[groupby]=="6",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [738]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.8
0    86571
4    61833
2    58016
3    52151
1    47554
5    20369
Name: count, dtype: int64

In [739]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#'Th22':#CCR4+ CCR6+ CCR10+ KLRB1- GZMK- GATA3lo CCL5- GATA3+,CCR4+ CCR6+ KLRB1- GZMKlo AHRlo
#'Th1/Th17':# CCR6+ CCR4- CXCR3+ KLRB1+ GZMK+ CCL5+
#'Th17':#step1 CCR6+ CCR4- KLRB1+ GZMK- CCL5- RORC+ GZMK- TBX21-
#'Th2':#step1 CCR4+ CCR6- KLRB1- GATA3+ GZMK- CCL5-；CCR4+ CXCR5- GATA3+ CCL5-.CD45RA- CD279- TBX21- LEF1+ ,且没有CCL5- GZMH- GZMK- GZMB- KLRB1-各种标记的情况下CD62L+ CD27+ CD25+ 
#'Th1':#step3 确认 CXCR3+ CCR6- KLRB1- GZMK+ CD279+ GZMK+ CCL5+. CD279+(核心) GZMK+(核心) TBX21+ CCL5+(核心) TIGIT+ KLRB1-(核心) CD127-/IL7Rlo CCR7-CD197- SELLlo/CD62Llo  GZMH- GNLY- PRF1- GZMB- 的是Th1

cell_dict = {'Th22':['3',],#CCR4+ CCR6+ CCR10+ KLRB1lo/- GZMK-    CRIP1 LGALS1 LGALS3 PI16 ANXA5 ANXA2
             'Th2':['0'],#PTGDR2=CRTH2 GATA3++ IL4R+ CCR4+ CCR6- KLRB1- GZMK- CCL5- CD62L+ CD25+     PTGDR2,SNED1,NEFL,GATA3,FXYD7,C1orf162
             #'Th1':['',],#EOMES+ GZMK+ CCL5+ KLRB1- CXCR3+ CCR6- CD279+ TBX21+ IFNG+.  CMC1,CST7,FCRL3,CCL4,SLAMF7,EOMES,PDCD1,NKG7,CCR5,KLRK1,F2R,PLEK
             'Th17':['1','4','5'],#CCR6=CD196++ RORC+ CCR4- KLRB1+ GZMK- CCL5- ICOS+    高表达LTK,PTPN13,PDE4D,CCR6,RORC,NR1D1,CTSH,KIF5C,LGALS3,USP10,CMTM6,TOB1
             'Th1/Th17':['2',],#DPP4+ CCR6+ EOMESlo CCR6lo CCR4- CXCR3+ KLRB1+ GZMKlo/+ CCL5+ TBX21+ 与Th1相似, 差异基因中等表达
             #'Doublet|Lowquality':[],
            }

In [740]:
check_dict_duplicates(cell_dict)

'无重复'

In [741]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R4'] = i

In [742]:
(adata.obs['Celltype_L4_L5_Refine_R4'].isna()).value_counts()

Celltype_L4_L5_Refine_R4
False    326494
Name: count, dtype: int64

In [743]:
adata.obs['Celltype_L4_L5_Refine_R4'].value_counts()

Celltype_L4_L5_Refine_R4
Th17        129756
Th2          86571
Th1/Th17     58016
Th22         52151
Name: count, dtype: int64

In [744]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine',
                           'Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3','Celltype_L4_L5_Refine_R4','receptor_type','receptor_type_BCR']]
indices.to_csv(f"{obj_path}/{celltype}/R7_refine_indices_{celltype}.csv")

# 3. Th17 T Cell Refine

In [781]:
celltype="Th17"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [782]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4,
           )

In [783]:
groupby = "L4_leiden_TOTALVI_0.5"

In [335]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [337]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='obs',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
sc.pl.umap(adt, color='HLA-DR',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',layer='dsb')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='CD183',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD185',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False,layer='dsb')
sc.pl.umap(adt, color='CD196',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False,layer='dsb')
sc.pl.umap(adt, color='CD161',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False,layer='dsb')
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD16',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False,layer='dsb')
sc.pl.umap(adt, color='IgM',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False,layer='dsb')
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='Celltype_L4_L5_Refine_R2',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='IFNG',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adata, color='IL6R',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
sc.pl.umap(adata, color='CCR10',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
#Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LTK','PTPN13','PDE4D','CCR6','RORC','NR1D1','CTSH','KIF5C','LGALS3','USP10','CMTM6','TOB1','MAF',
                     'TNFSF13B','CISH','AQP3','AUTS2','NSG1','S100A4'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1/Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'GZMH','IL18RAP','S1PR5','LYAR','NKG7','CST7','PRF1','TBX21','LINC01871','KLRG1','MYBL1','EOMES',
                     'EFHD2','DUSP2','SAMD3','CTSW','ID2','MATK','HOPX',],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','TBX21','IFNG',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th22
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','AHR',
                      'CRIP1','LGALS1','LGALS1','S100A10','S100A4','PI16','LMNA','ANXA5','ANXA2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th2
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'PTGDR2','SNED1','NEFL','GATA3','FXYD7','C1orf162','GDPD5','IL4R','CAPG',
                     'LGALS1','TNFSF10','TNFRSF4','PPP1R9B','CSGALNACT1','NIBAN1','ERN1','SORL1','RUNX2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th from BD
#TNFSF8=CD30L B3GAT1=CD57  BTLA= CD272  SLAMF5=CD84 HAVCR2=CD365
sc.pl.dotplot(adata, ['CXCR5','IL6R','TNFSF8','NRP1','IL21R','B3GAT1','BCL6','MAF','STAT3','ICOS','PDCD1','TIGIT','BTLA','CD200','SLAMF1','CD84',#Tfh
                      'IL4','IL17F','IL17A','IL21',#tfh分泌
                      'GATA3','SMAD1','STAT6','SPI1','IRF4',#Th9
                      'IL9','IL10','CCL17','CCL22','TGFB1',#th9分泌
                      'HAVCR2','CXCR4','CCR3','CCR4','CCR8','PTGDR2','GATA3','STAT5A','STAT6','MAF','GFI1','IRF4','NOTCH1','NOTCH2','IL1RL1','IL17RB','IFNGR1','IFNGR2','TNFRSF8',#Th2
                      'IL2','IL5','IL6','IL10','IL13','IL31',#Th2分泌
                      'CCR4','CCR6','CCR10','AHR','PDGFRA','PDGFRB',#Th22
                      'IL22','TNF',#Th22分泌
                      'CXCR3','CCR5','KLRD1','TBX21','STAT1','STAT4','EOMES','RUNX3','FASLG','IL12RB1','IL12RB2','IL18R1','IL27RA','NOTCH3','TNFSF11','ICOS','HAVCR2','DPP4',#Th1
                      'LTB','LTA','PRF1','GZMB','GZMA','TNF','IFNG',#Th1分泌
                      'CCR4','CCR6','KLRB1','ICOS','HAVCR2','RORC','RORA','STAT3','RUNX1','BATF','IRF4','MAF','IL6R','IL13RA1','IL21R','IL23R',#Th17
                      'TNF','CCL20','IL17A','IL17F','IL21','IL22','IL24','IL26',#Th17分泌
                      ],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs[groupby]=="6",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [784]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.5
0    254621
2    129762
1    112816
Name: count, dtype: int64

In [785]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#'Th22':#CCR4+ CCR6+ CCR10+ KLRB1- GZMK- GATA3lo CCL5- GATA3+,CCR4+ CCR6+ KLRB1- GZMKlo AHRlo
#'Th1/Th17':# CCR6+ CCR4- CXCR3+ KLRB1+ GZMK+ CCL5+
#'Th17':#step1 CCR6+ CCR4- KLRB1+ GZMK- CCL5- RORC+ GZMK- TBX21-
#'Th2':#step1 CCR4+ CCR6- KLRB1- GATA3+ GZMK- CCL5-；CCR4+ CXCR5- GATA3+ CCL5-.CD45RA- CD279- TBX21- LEF1+ ,且没有CCL5- GZMH- GZMK- GZMB- KLRB1-各种标记的情况下CD62L+ CD27+ CD25+ 
#'Th1':#step3 确认 CXCR3+ CCR6- KLRB1- GZMK+ CD279+ GZMK+ CCL5+. CD279+(核心) GZMK+(核心) TBX21+ CCL5+(核心) TIGIT+ KLRB1-(核心) CD127-/IL7Rlo CCR7-CD197- SELLlo/CD62Llo  GZMH- GNLY- PRF1- GZMB- 的是Th1

cell_dict = {#'Th22':['',],#CCR4+ CCR6+ CCR10+ KLRB1lo/- GZMK-    CRIP1 LGALS1 LGALS3 PI16 ANXA5 ANXA2
             #'Th2|Th22':[''],#PTGDR2=CRTH2 GATA3++ IL4R+ CCR4+ CCR6- KLRB1- GZMK- CCL5- CD62L+ CD25+     PTGDR2,SNED1,NEFL,GATA3,FXYD7,C1orf162
             #'Th1':['',],#EOMES+ GZMK+ CCL5+ KLRB1- CXCR3+ CCR6- CD279+ TBX21+ IFNG+.  CMC1,CST7,FCRL3,CCL4,SLAMF7,EOMES,PDCD1,NKG7,CCR5,KLRK1,F2R,PLEK
             'Th17':['1','0'],#CCR6=CD196++ RORC+ CCR4- KLRB1+ GZMK- CCL5- ICOS+    高表达LTK,PTPN13,PDE4D,CCR6,RORC,NR1D1,CTSH,KIF5C,LGALS3,USP10,CMTM6,TOB1
             'Th1/Th17':['2',],#DPP4+ CCR6+ EOMESlo CCR6lo CCR4- CXCR3+ KLRB1+ GZMKlo/+ CCL5+ TBX21+ 与Th1相似, 差异基因中等表达
             #'Doublet|Lowquality':[],
            }

#0号是有疑问的, CXCR3- CXCR5+ CCR6+ KLRB1+ GZMK- CCL5- CD40LG-


In [786]:
check_dict_duplicates(cell_dict)

'无重复'

In [787]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R4'] = i

In [788]:
(adata.obs['Celltype_L4_L5_Refine_R4'].isna()).value_counts()

Celltype_L4_L5_Refine_R4
False    497199
Name: count, dtype: int64

In [789]:
adata.obs['Celltype_L4_L5_Refine_R4'].value_counts()

Celltype_L4_L5_Refine_R4
Th17        367437
Th1/Th17    129762
Name: count, dtype: int64

In [790]:
adata.obs.loc[adata.obs['L4_leiden_TOTALVI_0.3'] == 1,'Celltype_L4_L5_Refine_R4'] = 'Proliferative help memory CD4+ T'#CD38 high UMI

In [791]:
adata = adata[adata.obs['L4_leiden_TOTALVI_1'] != "3",:]
adata = adata[adata.obs['L4_leiden_TOTALVI_1'] != "6",:]
adata = adata[adata.obs['L4_leiden_TOTALVI_1'] != "8",:]

In [792]:
adata.obs['Celltype_L4_L5_Refine_R4'].value_counts()

Celltype_L4_L5_Refine_R4
Th17        361674
Th1/Th17    127765
Name: count, dtype: int64

In [756]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine',
                           'Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3','Celltype_L4_L5_Refine_R4','receptor_type','receptor_type_BCR']]
indices.to_csv(f"{obj_path}/{celltype}/R7_refine_indices_{celltype}.csv")

# 4. Th1_Th17 T Cell Refine

In [715]:
celltype="Th1_Th17"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [716]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4,
           )

In [717]:
groupby = "L4_leiden_TOTALVI_0.5"

In [435]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [453]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='obs',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
sc.pl.umap(adt, color='HLA-DR',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',layer='dsb')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='CD183',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD185',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False,layer='dsb')
sc.pl.umap(adt, color='CD196',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False,layer='dsb')
sc.pl.umap(adt, color='CD161',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False,layer='dsb')
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD16',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False,layer='dsb')
sc.pl.umap(adt, color='IgM',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False,layer='dsb')
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='Celltype_L4_L5_Refine_R2',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='IFNG',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
sc.pl.umap(adata, color='CCR10',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
#Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LTK','PTPN13','PDE4D','CCR6','RORC','NR1D1','CTSH','KIF5C','LGALS3','USP10','CMTM6','TOB1',
                     'TNFSF13B','CISH','AQP3','AUTS2','NSG1','S100A4'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1/Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'GZMH','IL18RAP','S1PR5','LYAR','NKG7','CST7','PRF1','TBX21','LINC01871','KLRG1','MYBL1','EOMES',
                     'EFHD2','DUSP2','SAMD3','CTSW','ID2','MATK','HOPX',],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','TBX21','IFNG',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th22
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','AHR',
                      'CRIP1','LGALS1','LGALS1','S100A10','S100A4','PI16','LMNA','ANXA5','ANXA2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th2
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'PTGDR2','SNED1','NEFL','GATA3','FXYD7','C1orf162','GDPD5','IL4R','CAPG',
                     'LGALS1','TNFSF10','TNFRSF4','PPP1R9B','CSGALNACT1','NIBAN1','ERN1','SORL1','RUNX2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th from BD
#TNFSF8=CD30L B3GAT1=CD57  BTLA= CD272  SLAMF5=CD84 HAVCR2=CD365
sc.pl.dotplot(adata, ['CXCR5','IL6R','TNFSF8','NRP1','IL21R','B3GAT1','BCL6','MAF','STAT3','ICOS','PDCD1','TIGIT','BTLA','CD200','SLAMF1','CD84',#Tfh
                      'IL4','IL17F','IL17A','IL21',#tfh分泌
                      'GATA3','SMAD1','STAT6','SPI1','IRF4',#Th9
                      'IL9','IL10','CCL17','CCL22','TGFB1',#th9分泌
                      'HAVCR2','CXCR4','CCR3','CCR4','CCR8','PTGDR2','GATA3','STAT5A','STAT6','MAF','GFI1','IRF4','NOTCH1','NOTCH2','IL1RL1','IL17RB','IFNGR1','IFNGR2','TNFRSF8',#Th2
                      'IL2','IL5','IL6','IL10','IL13','IL31',#Th2分泌
                      'CCR4','CCR6','CCR10','AHR','PDGFRA','PDGFRB',#Th22
                      'IL22','TNF',#Th22分泌
                      'CXCR3','CCR5','KLRD1','TBX21','STAT1','STAT4','EOMES','RUNX3','FASLG','IL12RB1','IL12RB2','IL18R1','IL27RA','NOTCH3','TNFSF11','ICOS','HAVCR2','DPP4',#Th1
                      'LTB','LTA','PRF1','GZMB','GZMA','TNF','IFNG',#Th1分泌
                      'CCR4','CCR6','KLRB1','ICOS','HAVCR2','RORC','RORA','STAT3','RUNX1','BATF','IRF4','MAF','IL6R','IL13RA1','IL21R','IL23R',#Th17
                      'TNF','CCL20','IL17A','IL17F','IL21','IL22','IL24','IL26',#Th17分泌
                      ],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs[groupby]=="6",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [666]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.5
3    84034
1    73657
0    70047
2    43923
Name: count, dtype: int64

In [667]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#'Th22':#CCR4+ CCR6+ CCR10+ KLRB1- GZMK- GATA3lo CCL5- GATA3+,CCR4+ CCR6+ KLRB1- GZMKlo AHRlo
#'Th1/Th17':# CCR6+ CCR4- CXCR3+ KLRB1+ GZMK+ CCL5+
#'Th17':#step1 CCR6+ CCR4- KLRB1+ GZMK- CCL5- RORC+ GZMK- TBX21-
#'Th2':#step1 CCR4+ CCR6- KLRB1- GATA3+ GZMK- CCL5-；CCR4+ CXCR5- GATA3+ CCL5-.CD45RA- CD279- TBX21- LEF1+ ,且没有CCL5- GZMH- GZMK- GZMB- KLRB1-各种标记的情况下CD62L+ CD27+ CD25+ 
#'Th1':#step3 确认 CXCR3+ CCR6- KLRB1- GZMK+ CD279+ GZMK+ CCL5+. CD279+(核心) GZMK+(核心) TBX21+ CCL5+(核心) TIGIT+ KLRB1-(核心) CD127-/IL7Rlo CCR7-CD197- SELLlo/CD62Llo  GZMH- GNLY- PRF1- GZMB- 的是Th1

cell_dict = {#'Th22':['',],#CCR4+ CCR6+ CCR10+ KLRB1lo/- GZMK-    CRIP1 LGALS1 LGALS3 PI16 ANXA5 ANXA2
             #'Th2|Th22':[''],#PTGDR2=CRTH2 GATA3++ IL4R+ CCR4+ CCR6- KLRB1- GZMK- CCL5- CD62L+ CD25+     PTGDR2,SNED1,NEFL,GATA3,FXYD7,C1orf162
             #'Th1':['',],#EOMES+ GZMK+ CCL5+ KLRB1- CXCR3+ CCR6- CD279+ TBX21+ IFNG+.  CMC1,CST7,FCRL3,CCL4,SLAMF7,EOMES,PDCD1,NKG7,CCR5,KLRK1,F2R,PLEK
             'Th17':['1'],#CCR6=CD196++ RORC+ CCR4- KLRB1+ GZMK- CCL5- ICOS+    高表达LTK,PTPN13,PDE4D,CCR6,RORC,NR1D1,CTSH,KIF5C,LGALS3,USP10,CMTM6,TOB1
             'Th1/Th17':['0','2','3'],#DPP4+ CCR6+ EOMESlo CCR6lo CCR4- CXCR3+ KLRB1+ GZMKlo/+ CCL5+ TBX21+ 与Th1相似, 差异基因中等表达
             #'Doublet|Lowquality':[],
            }

#0号是有疑问的, 表达KLRB1 CXCR3 GZMK CCL5，,但是不表达CCR4- CCR6-

In [668]:
check_dict_duplicates(cell_dict)

'无重复'

In [669]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R4'] = i

In [670]:
(adata.obs['Celltype_L4_L5_Refine_R4'].isna()).value_counts()

Celltype_L4_L5_Refine_R4
False    271661
Name: count, dtype: int64

In [671]:
adata.obs['Celltype_L4_L5_Refine_R4'].value_counts()

Celltype_L4_L5_Refine_R4
Th1/Th17    198004
Th17         73657
Name: count, dtype: int64

In [672]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine',
                           'Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3','Celltype_L4_L5_Refine_R4','receptor_type','receptor_type_BCR']]
indices.to_csv(f"{obj_path}/{celltype}/R7_refine_indices_{celltype}.csv")

# 合并所有数据

## PBMC et. al.

In [31]:
try:
    del indices_dict,combined_indices,Finnal_indices_dict,combined_Finnal_indices_dict,Processing_combined_indices
except:
    print("It is None!")

It is None!


## 终版

In [32]:
celltypes=['AtypicalB',
           'Basophil','CEACAM8_Pos_Neutrophil','CEACAM8_Neg_Neutrophil',
           'CytotoxicCD4',
           'CD8Tcm',
           'DC','DnT',
           'HSPC',
           'iNKT','Mast','Memory_B','MAIT',
           'Monocyte','NaiveB','NaiveCD4','NaiveCD8','NK', 
           'Non_NK_ILC','Plasma','Platelet',
           'ProliferativeT','TemCD8',
           'TregCD4','TregCD8','Tfh_Tcm',
           'Vd1','Vd2',
]

In [33]:
Finnal_indices_dict= []
for i in tqdm(celltypes):
    indices = pd.read_csv(f'{finnal_path}{i}/Finnal_indices_{i}.csv',index_col=0)
    Finnal_indices_dict.append(indices)

100%|██████████| 28/28 [00:56<00:00,  2.04s/it]


In [34]:
combined_Finnal_indices_dict = pd.concat(Finnal_indices_dict)

In [35]:
combined_Finnal_indices_dict['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
Tfh                                  1125265
Terminal effector CD4+ T              717617
Vδ2 GZMB+                             630298
Treg memory T                         239193
Vδ2 GZMK+                             224338
MAIT CD27+                            207265
Treg Naïve T                           76314
Vδ2 GZMK+ HLA-DR+                      39016
MAIT CD27-                             34101
Treg KLRB1+ T                          31848
Naïve Vδ1                              14980
Treg HLA-DR hi T                       14793
Temra CD4+ T                           14511
Terminal effector HLA-DRhi CD4+ T      12670
Vδ1 effector GZMK+                     10359
Vδ1 SOX4+                               6557
Vδ1 effector KLRC2+                     4313
Vδ1 SOX4+ CD279+                        3356
Treg CD8+                               2827
MAIT CD56+                               755
Name: count, dtype: int64

## 过程

In [36]:
%%bash
ls /home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R7/ | wc -l
ls /home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R7/*/R7_refine* | wc -l
ls /home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Finnal/ | wc -l

4
4
28


In [37]:
celltypes=['Th1','Th2_Th22','Th1_Th17','Th17'
          ]

In [38]:
indices_dict= []
for i in tqdm(celltypes):
    indices = pd.read_csv(f'{obj_path}{i}/R7_refine_indices_{i}.csv',index_col=0)
    indices_dict.append(indices)

100%|██████████| 4/4 [00:01<00:00,  2.65it/s]


In [39]:
Processing_combined_indices = pd.concat(indices_dict)

In [40]:
combined_indices = pd.concat([Processing_combined_indices,combined_Finnal_indices_dict])

In [41]:
combined_indices['Celltype_L1_L2'].isna().value_counts()

Celltype_L1_L2
False    52018365
Name: count, dtype: int64

In [42]:
combined_indices['Celltype_L1_L2_Refine'].isna().value_counts()

Celltype_L1_L2_Refine
False    52018365
Name: count, dtype: int64

In [43]:
combined_indices['Celltype_L2_L3_Refine'].isna().value_counts()

Celltype_L2_L3_Refine
False    52018365
Name: count, dtype: int64

In [44]:
combined_indices['Celltype_L3_L4_Refine'].isna().value_counts()

Celltype_L3_L4_Refine
True     37417894
False    14600471
Name: count, dtype: int64

In [45]:
combined_indices['Celltype_L4_L5_Refine'].isna().value_counts()

Celltype_L4_L5_Refine
True     42117434
False     9900931
Name: count, dtype: int64

In [46]:
combined_indices['Celltype_L4_L5_Refine_R2'].isna().value_counts()

Celltype_L4_L5_Refine_R2
True     43416820
False     8601545
Name: count, dtype: int64

In [47]:
combined_indices['Celltype_L4_L5_Refine_R3'].isna().value_counts()

Celltype_L4_L5_Refine_R3
True     47317963
False     4700402
Name: count, dtype: int64

In [48]:
combined_indices['Celltype_L4_L5_Refine_R4'].isna().value_counts()

Celltype_L4_L5_Refine_R4
True     50728339
False     1290026
Name: count, dtype: int64

In [49]:
combined_indices['Celltype_L4_L5_Refine_R4'].value_counts()

Celltype_L4_L5_Refine_R4
Th17                                565087
Th1/Th17                            439209
Th1                                 113581
Th2                                  86571
Th22                                 69523
Proliferative help memory CD4+ T     16055
Name: count, dtype: int64

## save

In [50]:
def save_to_indices(celltype,celltype_to_file):
    L2_refine_path_R8 = '/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R8'
    indices=combined_indices['Celltype_L4_L5_Refine_R4'].isin(celltype)
    indices_celltype=combined_indices[indices]
    print(f"{indices_celltype['Celltype_L4_L5_Refine_R4'].value_counts()}")
    os.makedirs(f'{L2_refine_path_R8}/{celltype_to_file}', exist_ok=True)
    indices_celltype.to_csv(f"{L2_refine_path_R8}/{celltype_to_file}/R8_indices_{celltype_to_file}.csv")

In [51]:
Processing_combined_indices['Celltype_L4_L5_Refine_R4'].value_counts()

Celltype_L4_L5_Refine_R4
Th17                                565087
Th1/Th17                            439209
Th1                                 113581
Th2                                  86571
Th22                                 69523
Proliferative help memory CD4+ T     16055
Name: count, dtype: int64

In [778]:
save_to_indices(celltype=['Th1'],celltype_to_file='Th1')

Celltype_L4_L5_Refine_R4
Th1    113581
Name: count, dtype: int64


In [695]:
save_to_indices(celltype=['Th17'],celltype_to_file='Th17')

Celltype_L4_L5_Refine_R4
Th17    565087
Name: count, dtype: int64


In [696]:
save_to_indices(celltype=['Th2'],celltype_to_file='Th2')

Celltype_L4_L5_Refine_R4
Th2    86571
Name: count, dtype: int64


In [697]:
save_to_indices(celltype=['Th22'],celltype_to_file='Th22')

Celltype_L4_L5_Refine_R4
Th22    69523
Name: count, dtype: int64


In [779]:
save_to_indices(celltype=['Th1/Th17'],celltype_to_file='Th1_Th17')

Celltype_L4_L5_Refine_R4
Th1/Th17    439209
Name: count, dtype: int64


## 加入新的细胞到 ProliferativeT

In [52]:
celltype=['Proliferative help memory CD4+ T']
L2_refine_path_R8 = '/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R8'
indices=combined_indices['Celltype_L4_L5_Refine_R4'].isin(celltype)
indices_celltype=combined_indices[indices]

In [53]:
indices_celltype_before = pd.read_csv(f"{finnal_path}ProliferativeT/Finnal_indices_ProliferativeT.csv",index_col=0)

In [54]:
indices_celltype_combined = pd.concat([indices_celltype,indices_celltype_before])

In [55]:
print(f"{indices_celltype_combined['Celltype_L4_L5_Refine_R4'].value_counts()}")
print(f"{indices_celltype_combined['Celltype_L4_L5_Refine_R2'].value_counts()}")
os.makedirs(f'{L2_refine_path_R8}/ProliferativeT', exist_ok=True)
indices_celltype_combined.to_csv(f"{L2_refine_path_R8}/ProliferativeT/R8_indices_ProliferativeT.csv")

Celltype_L4_L5_Refine_R4
Proliferative help memory CD4+ T    16055
Name: count, dtype: int64
Celltype_L4_L5_Refine_R2
Th22                                11965
Proliferative help memory CD4+ T    10449
Proliferative memory CD8+ T          8642
Proliferative Treg                   6560
Proliferative Temra CD8+ T           5517
Th1                                  2436
Th1/Th17                             1142
Proliferative Dn T                    688
Proliferative Cytotoxic CD4+ T        316
Proliferative γδT T                   279
Th17                                  270
Th17(fromTreg)                        109
Th2(fromTreg)                          93
Th2                                    40
Name: count, dtype: int64
